## Рубежный контроль №1
#### Ковалев А.В. ИУ5-63Б
### Загрузка необходимых библиотек

In [7]:
import numpy as np
import pandas as pd
import random
from sklearn.impute import SimpleImputer
from sklearn.impute import MissingIndicator


### Загрузка и первичный анализ данных

In [8]:
df = pd.read_csv('toy_dataset.csv', sep=",")
print('Кол-во строк: ', df.shape[0],'\n' ,'Кол-во столбцов: ', df.shape[1], sep = '')
print(df.dtypes)
print('Пропуски по столбцам:', df.isnull().sum())
df.head(8)

Кол-во строк: 150000
Кол-во столбцов: 6
Number       int64
City        object
Gender      object
Age          int64
Income     float64
Illness     object
dtype: object
Пропуски по столбцам: Number     0
City       0
Gender     0
Age        0
Income     0
Illness    0
dtype: int64


,Number,City,Gender,Age,Income,Illness
0,1,Dallas,Male,41,40367.0,No
1,2,Dallas,Male,54,45084.0,No
2,3,Dallas,Male,42,52483.0,No
3,4,Dallas,Male,40,40941.0,No
4,5,Dallas,Male,46,50289.0,No
5,6,Dallas,Female,36,50786.0,No
6,7,Dallas,Female,32,33155.0,No
7,8,Dallas,Male,39,30914.0,No


#### Пропуски не обнаружены, поэтому создадим их искусственно
### Создание пропусков

In [9]:
for _ in range(15000): 
    df.loc[random.randint(0, len(df) - 1), random.choice(df.columns)] = np.nan

### Повторный анализ данных

In [10]:
print('Кол-во строк: ', df.shape[0],'\n' ,'Кол-во столбцов: ', df.shape[1], sep = '')
print(df.dtypes)
print('Пропуски по столбцам:', df.isnull().sum())
df.head(8)

Кол-во строк: 150000
Кол-во столбцов: 6
Number     float64
City        object
Gender      object
Age        float64
Income     float64
Illness     object
dtype: object
Пропуски по столбцам: Number     2441
City       2518
Gender     2481
Age        2457
Income     2429
Illness    2554
dtype: int64


,Number,City,Gender,Age,Income,Illness
0,1.0,Dallas,Male,41.0,40367.0,No
1,2.0,Dallas,Male,54.0,45084.0,No
2,3.0,Dallas,Male,42.0,52483.0,No
3,4.0,Dallas,Male,40.0,40941.0,No
4,5.0,Dallas,Male,46.0,50289.0,No
5,6.0,Dallas,Female,36.0,50786.0,No
6,7.0,Dallas,Female,32.0,33155.0,No
7,8.0,Dallas,Male,39.0,30914.0,No


### Проведем обработку пропусков в колонке City. Предположим, что данный фактор является для нас ключевым. Следовательно строки с пропуском в данной колонке не являются для нас полезными. Значит их можно удалить.

In [11]:
df.dropna(subset = ['City'])

,Number,City,Gender,Age,Income,Illness
0,1.0,Dallas,Male,41.0,40367.0,No
1,2.0,Dallas,Male,54.0,45084.0,No
2,3.0,Dallas,Male,42.0,52483.0,No
3,4.0,Dallas,Male,40.0,40941.0,No
4,5.0,Dallas,Male,46.0,50289.0,No
...,...,...,...,...,...,...
149995,149996.0,Austin,Male,48.0,93669.0,No
149996,149997.0,Austin,Male,25.0,96748.0,No
149997,149998.0,Austin,Male,26.0,111885.0,No
149998,149999.0,Austin,Male,25.0,111878.0,No


### Теперь заполним пропуски в Income средним значением по City

In [15]:
imputer = SimpleImputer(strategy = 'mean')
indicator = MissingIndicator()
df_initial_segments = [df[df['City'] == city] for city in df["City"].unique()]
df_segments = list()
for df_segment in df_initial_segments:
    if df_segment.shape[0]>0:
        df_segments.append(df_segment)
for i in range(len(df_segments)):
    score_col = df_segments[i][['Income']].copy()
    mask_missing= indicator.fit_transform(score_col)
    score_col = imputer.fit_transform(score_col)
    filled_df = score_col[mask_missing]
    print(filled_df[filled_df.size-1])
    print("!!!")
    df_segments[i]['Income'] = score_col
df_1 = pd.concat(df_segments)
print('Пропуски по столбцам:', df_1.isnull().sum())

45258.625032841155
!!!
96853.33647087838
!!!
95272.4376104063
!!!
135077.61891950847
!!!
91548.92510247174
!!!
70986.75171974523
!!!
100745.73599320883
!!!
90274.19463820489
!!!
Пропуски по столбцам: Number     2401
City          0
Gender     2437
Age        2412
Income        0
Illness    2511
dtype: int64


C:\Users\Andrew2003\AppData\Local\Temp\ipykernel_10356\839728370.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_segments[i]['Income'] = score_col
C:\Users\Andrew2003\AppData\Local\Temp\ipykernel_10356\839728370.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_segments[i]['Income'] = score_col
C:\Users\Andrew2003\AppData\Local\Temp\ipykernel_10356\839728370.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer